# TRELLIS.2 — image to 3D (managed model server)

**Kernel:** \`phase2_3d\`. This notebook is a Gradio client, not a model installer. It does not build native extensions, change PyTorch, or load TRELLIS weights into the notebook process.

Start the model server from the OEC repository root on an admitted idle A100:
\`\`\`bash
python tools/oec_service_manager.py start trellis-api --gpu <ID> --port 7868 --allow-download
python tools/oec_service_manager.py status trellis-api
\`\`\`
Use \`--allow-download\` only when a missing checkpoint needs to be provisioned and StorageManager gives fresh quota admission. The server binds to loopback; connect remotely through an authenticated SSH tunnel. The notebook validates the downloaded GLB header, embedded JSON chunk, mesh/node metadata and SHA-256.

In [ ]:
import json
import os
import urllib.parse
import requests

TRELLIS_API_BASE = os.environ.get("OEC_TRELLIS_API_URL", "http://127.0.0.1:7868").rstrip("/")
if urllib.parse.urlsplit(TRELLIS_API_BASE).hostname not in {"127.0.0.1", "localhost", "::1"}:
    raise RuntimeError("TRELLIS notebook only permits a loopback model-server endpoint.")
TRELLIS_HTTP = requests.Session()
TRELLIS_HTTP.trust_env = False

def trellis_service_health():
    try:
        response = TRELLIS_HTTP.get(f"{TRELLIS_API_BASE}/health", timeout=8)
        payload = response.json()
    except Exception as exc:
        raise gr.Error(
            f"TRELLIS model server is unreachable at {TRELLIS_API_BASE}. "
            "Start trellis-api on an admitted idle A100 and inspect service logs."
        ) from exc
    if response.status_code != 200 or not payload.get("ready") or not payload.get("model_loaded"):
        raise gr.Error(f"TRELLIS model server is not ready: {json.dumps(payload, ensure_ascii=False)}")
    return payload

In [ ]:
import base64
import hashlib
import json
import os
import struct
import time
from pathlib import Path

import gradio as gr
from PIL import Image

TRELLIS_RESULT_ROOT = Path("/tmp") / "oec-notebook-results"
TRELLIS_RESULT_ROOT.mkdir(parents=True, exist_ok=True)

def validate_glb_file(path: Path):
    data = path.read_bytes()
    if len(data) < 1024 or data[:4] != b"glTF":
        raise gr.Error("TRELLIS returned a missing, empty, or non-GLB asset.")
    version, declared_length = struct.unpack_from("<II", data, 4)
    if version != 2 or declared_length != len(data):
        raise gr.Error(f"Invalid GLB header: version={version}, declared={declared_length}, actual={len(data)}.")
    json_length, chunk_type = struct.unpack_from("<II", data, 12)
    if chunk_type != 0x4E4F534A or json_length <= 0 or 20 + json_length > len(data):
        raise gr.Error("Invalid GLB: first chunk is not a complete JSON chunk.")
    try:
        document = json.loads(data[20:20 + json_length].rstrip(b" \t\r\n\0").decode("utf-8"))
    except Exception as exc:
        raise gr.Error("TRELLIS GLB JSON chunk cannot be decoded.") from exc
    if not isinstance(document, dict) or "asset" not in document:
        raise gr.Error("TRELLIS GLB JSON chunk lacks the required glTF asset object.")
    return {
        "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(),
        "glb_version": version, "declared_length": declared_length,
        "json_chunk_bytes": json_length, "mesh_count": len(document.get("meshes", [])),
        "node_count": len(document.get("nodes", [])),
    }

def generate_3d(image_path, prompt, max_image_size, texture_size, decimation_target, remesh):
    if not image_path:
        raise gr.Error("Upload an input image.")
    path = Path(str(image_path)).expanduser().resolve()
    if not path.is_file() or path.is_symlink():
        raise gr.Error("Input image path is missing or unsafe.")
    health = trellis_service_health()
    raw_image = path.read_bytes()
    if not raw_image or len(raw_image) > 12 * 1024 * 1024:
        raise gr.Error("Input image must be non-empty and no larger than 12 MiB.")
    try:
        with Image.open(path) as probe:
            if probe.format not in {"PNG", "JPEG", "WEBP"}:
                raise ValueError(f"unsupported image format: {probe.format}")
            probe.verify()
    except Exception as exc:
        raise gr.Error("Input must be a valid PNG, JPEG or WEBP image.") from exc
    payload = {
        "task": "image_to_3d",
        "prompt": (prompt or "Generate a clean, detailed 3D asset").strip()[:800],
        "filename": path.name,
        "parameters": {
            "max_image_size": int(max_image_size), "texture_size": int(texture_size),
            "decimation_target": int(decimation_target), "remesh": bool(remesh),
        },
        "image_base64": base64.b64encode(raw_image).decode("ascii"),
    }
    response = TRELLIS_HTTP.post(f"{TRELLIS_API_BASE}/v1/jobs", json=payload, timeout=40)
    if response.status_code != 202:
        raise gr.Error(f"TRELLIS job rejected (HTTP {response.status_code}): {response.text[:1200]}")
    accepted = response.json()
    job_id = accepted.get("job_id")
    if not isinstance(job_id, str) or not job_id.startswith("job_"):
        raise gr.Error(f"Unexpected TRELLIS job response: {accepted}")
    deadline = time.monotonic() + 1800
    latest = accepted
    while time.monotonic() < deadline:
        poll = TRELLIS_HTTP.get(f"{TRELLIS_API_BASE}/v1/jobs/{job_id}", timeout=20)
        if poll.status_code != 200:
            raise gr.Error(f"TRELLIS job status lookup failed (HTTP {poll.status_code}): {poll.text[:800]}")
        latest = poll.json()
        status = latest.get("status")
        if status == "succeeded":
            break
        if status in {"failed", "error", "cancelled"}:
            raise gr.Error(f"TRELLIS job {status}: {latest.get('message', 'No error details returned')}")
        time.sleep(2)
    else:
        raise gr.Error("TRELLIS inference timed out; inspect trellis-api service logs.")
    result = TRELLIS_HTTP.get(f"{TRELLIS_API_BASE}/v1/jobs/{job_id}/result", timeout=180)
    if result.status_code != 200:
        raise gr.Error(f"TRELLIS asset retrieval failed (HTTP {result.status_code}): {result.text[:800]}")
    content_type = result.headers.get("Content-Type", "").split(";", 1)[0].lower()
    if content_type not in {"model/gltf-binary", "application/octet-stream", "model/glb"}:
        raise gr.Error(f"Expected a binary GLB, received Content-Type={content_type!r}.")
    output_path = TRELLIS_RESULT_ROOT / f"{job_id}.glb"
    if output_path.exists():
        raise gr.Error("Refusing to overwrite an existing notebook result.")
    output_path.write_bytes(result.content)
    try:
        validation = validate_glb_file(output_path)
    except Exception:
        output_path.unlink(missing_ok=True)
        raise
    details = {
        "job_id": job_id, "status": latest.get("status"), "api": TRELLIS_API_BASE,
        "health": health, "metadata": latest.get("metadata", {}),
        "artifact_path": str(output_path), "validation": validation,
        "input_filename": path.name, "license_boundary": "TRELLIS.2 terms must be reviewed for intended use",
    }
    return str(output_path), json.dumps(details, indent=2, ensure_ascii=False)

with gr.Blocks(title="TRELLIS.2 — managed model server") as demo_trellis:
    gr.Markdown("# TRELLIS.2 — image to 3D")
    gr.Markdown("Inference and model weights are managed by the separate trellis-api process. The notebook downloads and validates the GLB returned by the API.")
    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="Input image", type="filepath")
            prompt = gr.Textbox(label="Prompt metadata (not model-conditioned)", value="A clean, detailed 3D asset", lines=2)
            max_image_size = gr.Dropdown([512, 768, 1024, 1280], value=512, label="Maximum input image size")
            texture_size = gr.Dropdown([512, 1024, 2048, 4096], value=512, label="Texture size")
            decimation_target = gr.Slider(1000, 500000, value=100000, step=1000, label="Decimation target")
            remesh = gr.Checkbox(value=False, label="Remesh")
            generate_button = gr.Button("Generate 3D model", variant="primary")
        with gr.Column():
            output_model = gr.Model3D(label="Validated GLB")
            output_metadata = gr.Textbox(label="Job and GLB metadata", lines=14)
    generate_button.click(
        fn=generate_3d,
        inputs=[input_image, prompt, max_image_size, texture_size, decimation_target, remesh],
        outputs=[output_model, output_metadata],
    )

if os.environ.get("OEC_GRADIO_HEADLESS_SMOKE") != "1":
    demo_trellis.queue(max_size=1).launch(
        server_name="127.0.0.1", server_port=7862, inline=False, share=False, show_error=True, allowed_paths=[str(TRELLIS_RESULT_ROOT)]
    )